# ⚗️ CatalystAI — AI Co-Scientist for Sustainable Fuel Discovery
> **Powered by Google Gemini &nbsp;·&nbsp; Generative · Predictive · Explainable · Self-Learning**

Run each cell in order. The final cell launches the full interactive UI.

In [7]:
# ── CELL 1 · Install dependencies ──────────────────────────────────────────
!pip install -q google-generativeai openai gradio plotly pandas scikit-learn numpy matplotlib
print('✅ All packages installed.')

✅ All packages installed.


In [8]:
import os, getpass
from openai import OpenAI

if 'OPENROUTER_API_KEY' not in os.environ:
    os.environ['OPENROUTER_API_KEY'] = getpass.getpass("Enter OpenRouter API Key: ")

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ['OPENROUTER_API_KEY']
)

MODEL_NAME = "openrouter/free"

try:
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {"role": "user", "content": "Say OK"}
        ],
        max_tokens=10
    )

    print("✅ Connected Successfully")
    print(response.choices[0].message.content)

except Exception as e:
    print("❌ Connection failed:", e)

✅ Connected Successfully
OK


In [9]:
import json, random, math, re, textwrap, time
from dataclasses import dataclass, field, asdict
from typing import List, Dict, Any, Optional
import numpy as np
import pandas as pd
from openai import OpenAI

# ─────────────────────────────────────────────
#  Data model
# ─────────────────────────────────────────────
@dataclass
class CatalystCandidate:
    name: str
    composition: str
    support: str
    dopants: List[str]
    synthesis_route: str
    activity_score: float
    stability_score: float
    selectivity_score: float
    cost_score: float
    uncertainty: float
    explanation: str
    key_features: List[str]
    pareto_rank: int = 0
    overall_score: float = 0.0
    novel: bool = True

@dataclass
class ExperimentalResult:
    catalyst_name: str
    measured_activity: float
    measured_stability: float
    notes: str
    timestamp: str = field(default_factory=lambda: time.strftime('%Y-%m-%d %H:%M'))

# ─────────────────────────────────────────────
#  Persistent memory
# ─────────────────────────────────────────────
feedback_db: List[ExperimentalResult] = []
chat_history: List[Dict] = []
all_sessions: List[Dict] = []


# ─────────────────────────────────────────────
#  OpenRouter Client
# ─────────────────────────────────────────────
client = OpenAI(
    base_url='https://openrouter.ai/api/v1',
    api_key=os.environ['OPENROUTER_API_KEY'],
)

MODEL_NAME = 'deepseek/deepseek-chat-v3-0324:free'


# ─────────────────────────────────────────────
#  LLM helper
# ─────────────────────────────────────────────
def ask_llm(prompt: str, system_instruction: str = '') -> str:
    try:
        messages = []

        if system_instruction:
            messages.append({
                'role': 'system',
                'content': system_instruction
            })

        messages.append({
            'role': 'user',
            'content': prompt
        })

        response = client.chat.completions.create(
            model=MODEL_NAME,
            messages=messages,
            temperature=0.7,
            max_tokens=3000,
        )

        return response.choices[0].message.content

    except Exception as e:
        return f'LLM Error: {str(e)}'

# ─────────────────────────────────────────────
#  Candidate generation
# ─────────────────────────────────────────────
def generate_candidates(
    reaction: str,
    temperature: float,
    pressure: float,
    solvent: str,
    excluded: str,
    constraints: str,
    n_candidates: int = 5,
):

    prompt = f'''
You are an advanced AI catalyst discovery system.

Generate {n_candidates} catalyst candidates.

Reaction Goal:
{reaction}

Conditions:
- Temperature: {temperature} C
- Pressure: {pressure} atm
- Solvent: {solvent}

Avoid:
{excluded}

Constraints:
{constraints}

Return STRICT JSON list.

Format:
[
  {{
    "name": "Catalyst Name",
    "composition": "NiFeOx",
    "support": "Al2O3",
    "dopants": ["Cu", "Mo"],
    "synthesis_route": "Hydrothermal synthesis",
    "activity_score": 88,
    "stability_score": 80,
    "selectivity_score": 90,
    "cost_score": 76,
    "uncertainty": 0.15,
    "explanation": "Reasoning here",
    "key_features": ["oxygen vacancies", "high surface area"]
  }}
]
'''

    raw = ask_llm(
        prompt,
        system_instruction='You are an expert catalyst scientist and materials AI researcher.'
    )

    try:
        json_match = re.search(r'\[.*\]', raw, re.DOTALL)

        if json_match:
            parsed = json.loads(json_match.group())
        else:
            parsed = json.loads(raw)

        candidates = []

        for item in parsed:
            cand = CatalystCandidate(
                name=item['name'],
                composition=item['composition'],
                support=item['support'],
                dopants=item['dopants'],
                synthesis_route=item['synthesis_route'],
                activity_score=float(item['activity_score']),
                stability_score=float(item['stability_score']),
                selectivity_score=float(item['selectivity_score']),
                cost_score=float(item['cost_score']),
                uncertainty=float(item['uncertainty']),
                explanation=item['explanation'],
                key_features=item['key_features'],
            )

            cand.overall_score = (
                cand.activity_score * 0.35 +
                cand.stability_score * 0.25 +
                cand.selectivity_score * 0.25 +
                cand.cost_score * 0.15
            )

            candidates.append(cand)

        candidates.sort(key=lambda x: x.overall_score, reverse=True)

        for i, c in enumerate(candidates):
            c.pareto_rank = i + 1

        return candidates

    except Exception as e:
        print('Parsing error:', e)
        print(raw)
        return []

# ─────────────────────────────────────────────
#  Chat with AI
# ─────────────────────────────────────────────
def chat_with_ai(message: str, candidates: list) -> str:
    context = ""
    if candidates:
        context = "\n\nCurrent candidate list:\n"
        for c in candidates:
            context += f"- {c.name}: activity={c.activity_score:.1f}, stability={c.stability_score:.1f}, overall={c.overall_score:.1f}\n"

    if feedback_db:
        context += "\n\nExperimental feedback so far:\n"
        for r in feedback_db[-5:]:
            context += f"- {r.catalyst_name}: activity={r.measured_activity}, stability={r.measured_stability}, notes={r.notes}\n"

    return ask_llm(
        prompt=message + context,
        system_instruction=(
            "You are CatalystAI, an expert AI co-scientist for catalyst discovery. "
            "Answer concisely and scientifically. Reference the candidate list when relevant."
        )
    )


print('✅ Core logic + chat helper loaded.')


✅ Core logic + chat helper loaded.


In [10]:
# ── CELL 4 · Visualisation helpers ─────────────────────────────────────────
import plotly.graph_objects as go
import plotly.express as px

PALETTE = [
    '#00d4aa', '#ff6b6b', '#ffd166', '#06d6a0',
    '#118ab2', '#e76f51', '#a8dadc', '#f4a261'
]
BG     = '#0d1117'
PANEL  = '#161b22'
TEXT   = '#c9d1d9'
ACCENT = '#00d4aa'

LAYOUT_BASE = dict(
    paper_bgcolor=BG, plot_bgcolor=PANEL,
    font=dict(color=TEXT, family='monospace'),
    margin=dict(l=40, r=40, t=50, b=40),
)


def make_radar(candidates: List[CatalystCandidate]) -> go.Figure:
    categories = ['Activity', 'Stability', 'Selectivity', 'Cost', 'Confidence']
    fig = go.Figure()
    for i, c in enumerate(candidates[:6]):
        conf = (1 - c.uncertainty) * 100
        vals = [c.activity_score, c.stability_score, c.selectivity_score, c.cost_score, conf]
        fig.add_trace(go.Scatterpolar(
            r=vals + [vals[0]],
            theta=categories + [categories[0]],
            name=c.name,
            line=dict(color=PALETTE[i % len(PALETTE)], width=2),
            fill='toself',
            opacity=0.85,
        ))
    fig.update_layout(
        **LAYOUT_BASE,
        title=dict(text='📡 Multi-Objective Radar', font=dict(color=ACCENT, size=16)),
        polar=dict(
            bgcolor=PANEL,
            radialaxis=dict(visible=True, range=[0, 100], color=TEXT, gridcolor='#30363d'),
            angularaxis=dict(color=TEXT, gridcolor='#30363d'),
        ),
        legend=dict(bgcolor='rgba(0,0,0,0)'),
        height=450,
    )
    return fig


def make_pareto(candidates):

    if not candidates:
        fig = go.Figure()

        fig.update_layout(
            title="No candidates generated",
            paper_bgcolor="#0d1117",
            plot_bgcolor="#161b22",
            font=dict(color="white")
        )

        return fig

    df = pd.DataFrame([
        {
            'name': c.name,
            'activity': c.activity_score,
            'stability': c.stability_score,
            'overall': c.overall_score,
        }
        for c in candidates
    ])

    fig = px.scatter(
        df,
        x='activity',
        y='stability',
        size='overall',
        hover_name='name',
        title='Pareto Frontier'
    )

    fig.update_layout(
        paper_bgcolor="#0d1117",
        plot_bgcolor="#161b22",
        font=dict(color="white")
    )

    return fig


def make_bar(candidates: List[CatalystCandidate]) -> go.Figure:
    names = [c.name for c in candidates]
    fig = go.Figure()
    metrics = [
        ('Activity',    [c.activity_score    for c in candidates], PALETTE[0]),
        ('Stability',   [c.stability_score   for c in candidates], PALETTE[1]),
        ('Selectivity', [c.selectivity_score for c in candidates], PALETTE[2]),
        ('Cost',        [c.cost_score        for c in candidates], PALETTE[3]),
    ]
    for label, vals, color in metrics:
        fig.add_trace(go.Bar(name=label, x=names, y=vals, marker_color=color))
    err = [c.uncertainty * 30 for c in candidates]
    fig.data[0].error_y = dict(type='data', array=err, visible=True, color='#ffffff80')
    fig.update_layout(
        **LAYOUT_BASE,
        title=dict(text='📊 Score Breakdown (error bars = uncertainty × 30)', font=dict(color=ACCENT, size=15)),
        barmode='group',
        xaxis=dict(gridcolor='#30363d', tickangle=-20),
        yaxis=dict(gridcolor='#30363d', title='Score (0-100)'),
        legend=dict(bgcolor='rgba(0,0,0,0)'),
        height=400,
    )
    return fig


def make_uncertainty(candidates: List[CatalystCandidate]) -> go.Figure:
    df = pd.DataFrame([
        dict(name=c.name, overall=c.overall_score, uncertainty=c.uncertainty,
             conf=f"{(1-c.uncertainty)*100:.0f}%")
        for c in candidates
    ])
    fig = go.Figure()
    colors = [PALETTE[i % len(PALETTE)] for i in range(len(df))]
    fig.add_trace(go.Bar(
        x=df.name, y=df.uncertainty,
        marker_color=colors,
        text=df.conf, textposition='outside',
        textfont=dict(color=TEXT),
    ))
    fig.add_hline(y=0.3, line_dash='dash', line_color='#ff6b6b',
                  annotation_text='Caution threshold', annotation_font_color='#ff6b6b')
    fig.update_layout(
        **LAYOUT_BASE,
        title=dict(text='🎲 Prediction Uncertainty (lower = more confident)', font=dict(color=ACCENT, size=15)),
        xaxis=dict(gridcolor='#30363d', tickangle=-20),
        yaxis=dict(gridcolor='#30363d', title='Uncertainty (0–1)', range=[0, 1.1]),
        height=380,
    )
    return fig


def make_feedback_trend() -> go.Figure:
    if not feedback_db:
        fig = go.Figure()
        fig.add_annotation(
            text='No experimental data yet.\nSubmit results in the Feedback tab.',
            xref='paper', yref='paper', x=0.5, y=0.5, showarrow=False,
            font=dict(color=TEXT, size=14)
        )
        fig.update_layout(**LAYOUT_BASE, height=350)
        return fig
    df = pd.DataFrame([asdict(r) for r in feedback_db])
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=list(range(len(df))), y=df.measured_activity,
        mode='lines+markers', name='Measured Activity',
        line=dict(color=PALETTE[0], width=2), marker=dict(size=8),
        text=df.catalyst_name, hovertemplate='%{text}<br>Activity: %{y:.1f}',
    ))
    fig.add_trace(go.Scatter(
        x=list(range(len(df))), y=df.measured_stability,
        mode='lines+markers', name='Measured Stability',
        line=dict(color=PALETTE[1], width=2), marker=dict(size=8),
        text=df.catalyst_name, hovertemplate='%{text}<br>Stability: %{y:.1f}',
    ))
    fig.update_layout(
        **LAYOUT_BASE,
        title=dict(text='🔬 Self-Learning: Experimental Feedback Loop', font=dict(color=ACCENT, size=15)),
        xaxis=dict(title='Experiment #', gridcolor='#30363d'),
        yaxis=dict(title='Measured Score', gridcolor='#30363d'),
        height=380,
    )
    return fig


print('✅ Visualisation helpers loaded.')

✅ Visualisation helpers loaded.


In [11]:
# ── CELL 5 · Gradio UI ─────────────────────────────────────────────────────
import gradio as gr

# shared state
_candidates: List[CatalystCandidate] = []


# ─── helpers ───

def candidates_to_df(cands: List[CatalystCandidate]) -> pd.DataFrame:
    if not cands:
        return pd.DataFrame()
    rows = []
    for c in cands:
        rows.append({
            'Rank': c.pareto_rank,
            'Name': c.name,
            'Composition': c.composition,
            'Support': c.support,
            'Activity': f'{c.activity_score:.1f}',
            'Stability': f'{c.stability_score:.1f}',
            'Selectivity': f'{c.selectivity_score:.1f}',
            'Cost': f'{c.cost_score:.1f}',
            'Confidence': f"{(1-c.uncertainty)*100:.0f}%",
            'Overall ▼': f'{c.overall_score:.1f}',
        })
    return pd.DataFrame(rows)


def xai_markdown(cands: List[CatalystCandidate]) -> str:
    if not cands:
        return '_Run a discovery first._'
    parts = []
    for c in cands:
        conf = (1 - c.uncertainty) * 100
        badge = '🟢' if conf >= 70 else ('🟡' if conf >= 45 else '🔴')
        parts.append(
            f"### {badge} {c.name}  ·  Rank #{c.pareto_rank}\n"
            f"**Composition:** `{c.composition}` / **Support:** `{c.support}`\n\n"
            f"**Dopants:** {', '.join(c.dopants) if c.dopants else 'None'}\n\n"
            f"**Synthesis:** {c.synthesis_route}\n\n"
            f"**AI Explanation:**\n> {c.explanation}\n\n"
            f"**Key Descriptors:** " + ' · '.join(f'`{f}`' for f in c.key_features) + "\n\n"
            f"**Confidence:** {conf:.0f}%  |  **Uncertainty:** {c.uncertainty:.3f}\n"
            + '---'
        )
    return '\n\n'.join(parts)


# ─── main generation handler ───

def run_discovery(reaction, temperature, pressure, solvent, excluded, constraints, n_candidates):
    global _candidates
    if not reaction.strip():
        return ('⚠️ Please enter a reaction goal.', None, None, None, None, None, None)
    try:
        cands = generate_candidates(
            reaction, temperature, pressure, solvent,
            excluded, constraints, int(n_candidates)
        )
        _candidates = cands
        all_sessions.append({
            'reaction': reaction,
            'candidates': [asdict(c) for c in cands],
            'timestamp': time.strftime('%Y-%m-%d %H:%M'),
        })
        df    = candidates_to_df(cands)
        xai   = xai_markdown(cands)
        radar = make_radar(cands)
        pareto= make_pareto(cands)
        bar   = make_bar(cands)
        unc   = make_uncertainty(cands)
        msg   = f'✅ {len(cands)} candidates generated and ranked by Gemini!'
        return msg, df, xai, radar, pareto, bar, unc
    except Exception as e:
        import traceback
        tb = traceback.format_exc()
        return f'❌ Error: {e}\n\n{tb}', None, None, None, None, None, None


# ─── chat handler ───

def chat_fn(message, history):
    return chat_with_ai(message, _candidates)


# ─── feedback handler ───

def submit_feedback(cat_name, act, stab, notes):
    if not cat_name.strip():
        return '⚠️ Enter a catalyst name.', make_feedback_trend()
    feedback_db.append(ExperimentalResult(
        catalyst_name=cat_name,
        measured_activity=float(act),
        measured_stability=float(stab),
        notes=notes,
    ))
    msg = f'✅ Feedback recorded for **{cat_name}**. Gemini will use this in future suggestions.'
    return msg, make_feedback_trend()


# ─── export handlers ───

def export_json():
    if not _candidates:
        return 'No candidates to export.'
    path = '/content/catalyst_results.json'
    with open(path, 'w') as f:
        json.dump([asdict(c) for c in _candidates], f, indent=2)
    return f'✅ Exported to {path}'


def export_csv():
    if not _candidates:
        return 'No candidates to export.'
    path = '/content/catalyst_results.csv'
    candidates_to_df(_candidates).to_csv(path, index=False)
    return f'✅ Exported to {path}'


# ─────────────────────────────────────────────
#  CSS  (dark sci-fi theme)
# ─────────────────────────────────────────────
CSS = """
:root {
  --bg:    #0d1117;
  --panel: #161b22;
  --border:#30363d;
  --text:  #c9d1d9;
  --accent:#00d4aa;
  --warn:  #ffd166;
  --danger:#ff6b6b;
  --radius:12px;
}
body, .gradio-container {
  background: var(--bg) !important;
  color: var(--text) !important;
  font-family: 'JetBrains Mono', 'Fira Code', monospace !important;
}
.gradio-container { max-width: 1280px !important; margin: auto; }

#catalyst-header {
  background: linear-gradient(135deg, #0d1117 0%, #1a2744 50%, #0d1117 100%);
  border: 1px solid var(--accent);
  border-radius: var(--radius);
  padding: 28px 36px;
  margin-bottom: 20px;
  text-align: center;
  position: relative;
  overflow: hidden;
}
#catalyst-header::before {
  content: '';
  position: absolute; inset: 0;
  background: radial-gradient(ellipse at 50% 0%, rgba(0,212,170,.18) 0%, transparent 70%);
  pointer-events: none;
}
#catalyst-header h1 {
  font-size: 2.4rem; font-weight: 800; letter-spacing: -1px;
  background: linear-gradient(90deg, #00d4aa, #06d6a0, #ffd166);
  -webkit-background-clip: text; -webkit-text-fill-color: transparent;
  margin: 0 0 6px;
}
#catalyst-header p { color: #8b949e; font-size: .95rem; margin: 0; }
#gemini-badge {
  display: inline-block; margin-top: 10px;
  background: linear-gradient(90deg, #4285f4, #34a853, #fbbc04, #ea4335);
  -webkit-background-clip: text; -webkit-text-fill-color: transparent;
  font-size: .85rem; font-weight: 700; letter-spacing: 1px;
}

.tab-nav button {
  background: var(--panel) !important;
  border: 1px solid var(--border) !important;
  color: var(--text) !important;
  border-radius: 8px !important;
  font-size: .88rem !important;
  margin: 2px !important;
}
.tab-nav button.selected {
  border-color: var(--accent) !important;
  color: var(--accent) !important;
  background: rgba(0,212,170,.1) !important;
}

.gr-box, .gr-form, .gradio-group {
  background: var(--panel) !important;
  border: 1px solid var(--border) !important;
  border-radius: var(--radius) !important;
}

input, textarea, select {
  background: var(--bg) !important;
  border: 1px solid var(--border) !important;
  color: var(--text) !important;
  border-radius: 8px !important;
}
input:focus, textarea:focus {
  border-color: var(--accent) !important;
  box-shadow: 0 0 0 2px rgba(0,212,170,.25) !important;
}

button.primary, .gr-button-primary {
  background: linear-gradient(135deg, #00d4aa, #06d6a0) !important;
  color: #0d1117 !important;
  font-weight: 700 !important;
  border: none !important;
  border-radius: 8px !important;
  letter-spacing: .5px;
}
button.secondary, .gr-button-secondary {
  background: var(--panel) !important;
  border: 1px solid var(--accent) !important;
  color: var(--accent) !important;
  border-radius: 8px !important;
}

label span, .gr-label { color: #8b949e !important; font-size: .82rem !important; }

#status-box textarea {
  background: rgba(0,212,170,.05) !important;
  border-color: var(--accent) !important;
  font-size: .9rem !important;
}

.dataframe th {
  background: var(--panel) !important;
  color: var(--accent) !important;
  border-bottom: 1px solid var(--border) !important;
}
.dataframe td {
  background: var(--bg) !important;
  color: var(--text) !important;
  border-bottom: 1px solid var(--border) !important;
}
.dataframe tr:hover td { background: rgba(0,212,170,.07) !important; }

.message.user { background: rgba(0,212,170,.12) !important; border-radius: 12px !important; }
.message.bot  { background: var(--panel) !important; border-radius: 12px !important; }

.prose h3 { color: var(--accent); border-bottom: 1px solid var(--border); padding-bottom: 4px; }
.prose code { background: rgba(0,212,170,.12); border-radius: 4px; padding: 1px 5px; color: var(--accent); }
.prose blockquote { border-left: 3px solid var(--accent); padding-left: 12px; color: #8b949e; }
"""

# ─────────────────────────────────────────────
#  Build UI
# ─────────────────────────────────────────────
EXAMPLE_REACTIONS = [
    'CO₂ hydrogenation to methanol, no platinum-group metals',
    'Green hydrogen production via water splitting',
    'Fischer-Tropsch synthesis for synthetic diesel',
    'Ammonia synthesis at low temperature and pressure',
    'Biomass pyrolysis oil upgrading',
    'Methane dry reforming with high coke resistance',
]

with gr.Blocks(css=CSS, title='CatalystAI — Gemini') as demo:

    # ── Header ──────────────────────────────────────
    gr.HTML("""
    <div id="catalyst-header">
      <h1>⚗️ CatalystAI</h1>
      <p>AI Co-Scientist for Sustainable Fuel Discovery &nbsp;·&nbsp;
         Generative &nbsp;·&nbsp; Predictive &nbsp;·&nbsp; Explainable &nbsp;·&nbsp; Self-Learning</p>
      <span id="gemini-badge">✦ Powered by Google Gemini 1.5 Pro ✦</span>
    </div>
    """)

    with gr.Tabs():

        # ── TAB 1: Discovery ──────────────────────────
        with gr.Tab('🔭 Discovery'):
            with gr.Row():
                with gr.Column(scale=4):
                    reaction_input = gr.Textbox(
                        label='🎯 Reaction Goal (natural language)',
                        placeholder='e.g. CO₂ hydrogenation to methanol, no platinum-group metals',
                        lines=2,
                    )
                    with gr.Row():
                        temp_input  = gr.Slider(100, 900, value=350, step=10, label='Temperature (°C)')
                        press_input = gr.Slider(1,   300, value=50,  step=1,  label='Pressure (bar)')
                    with gr.Row():
                        solvent_input = gr.Dropdown(
                            choices=['Gas phase', 'Aqueous', 'Organic (THF)', 'Supercritical CO₂', 'Ionic liquid', 'Ethanol'],
                            value='Gas phase', label='Solvent / Phase'
                        )
                        n_slider = gr.Slider(3, 10, value=6, step=1, label='# Candidates')
                    with gr.Row():
                        excl_input  = gr.Textbox(label='🚫 Excluded metals', placeholder='Pt, Pd, Rh …')
                        const_input = gr.Textbox(label='📌 Extra constraints', placeholder='earth-abundant only, low Ea …')
                    gen_btn = gr.Button('🚀 Generate Catalysts', variant='primary', size='lg')
                    gr.Examples(
                        examples=[[r, 350, 50, 'Gas phase', '', '', 6] for r in EXAMPLE_REACTIONS],
                        inputs=[reaction_input, temp_input, press_input, solvent_input,
                                excl_input, const_input, n_slider],
                        label='Quick examples',
                    )
                with gr.Column(scale=2):
                    status_box = gr.Textbox(
                        label='Status',
                        value='Ready. Enter a reaction goal and click Generate.',
                        lines=3, interactive=False, elem_id='status-box'
                    )
                    gr.Markdown('### 📚 Mission')
                    gr.Markdown(
                        'CatalystAI **generates** novel catalysts beyond any database, '
                        '**predicts** multi-objective scores with calibrated uncertainty, '
                        '**explains** decisions via XAI, and **learns** from your lab results.'
                    )
                    gr.Markdown('> 🤖 Reasoning engine: **Google Gemini 1.5 Pro**')

        # ── TAB 2: Results ────────────────────────────
        with gr.Tab('📋 Results Table'):
            results_df = gr.Dataframe(
                headers=['Rank','Name','Composition','Support','Activity',
                         'Stability','Selectivity','Cost','Confidence','Overall ▼'],
                label='Ranked Candidates',
                interactive=False,
                wrap=True,
            )
            with gr.Row():
                btn_json = gr.Button('⬇ Export JSON', variant='secondary')
                btn_csv  = gr.Button('⬇ Export CSV',  variant='secondary')
            export_status = gr.Textbox(label='Export status', interactive=False, lines=1)

        # ── TAB 3: XAI Explanations ───────────────────
        with gr.Tab('🧪 XAI Explanations'):
            xai_md = gr.Markdown('_Run a discovery first._')

        # ── TAB 4: Visualisations ─────────────────────
        with gr.Tab('📊 Visualisations'):
            with gr.Row():
                radar_plot = gr.Plot(label='Radar')
                bar_plot   = gr.Plot(label='Bar')
            with gr.Row():
                pareto_plot = gr.Plot(label='Pareto')
                unc_plot    = gr.Plot(label='Uncertainty')

        # ── TAB 5: AI Chat ────────────────────────────
        with gr.Tab('💬 AI Chat'):
            gr.Markdown(
                'Ask CatalystAI anything — compare candidates, request synthesis details, '
                'explain descriptors, suggest experiments, or discuss catalytic mechanisms. '
                '**Powered by Gemini multi-turn conversation.**'
            )
            chatbot = gr.ChatInterface(
                fn=chat_fn,
                examples=[
                    'Which candidate has the best trade-off between cost and activity?',
                    'Explain the d-band theory and how it applies to the top candidate.',
                    'What synthesis steps would you recommend for the #1 candidate?',
                    'Why does high uncertainty appear for exotic compositions?',
                    'How does CO₂ adsorption affect methanol selectivity?',
                ],
                type='messages',
            )

        # ── TAB 6: Feedback / Self-Learning ───────────
        with gr.Tab('🔬 Feedback Loop'):
            gr.Markdown(
                '### 🔄 Close the Loop\n'
                'Submit your experimental results. CatalystAI will inject this data into '
                'every future Gemini prompt, producing better-calibrated suggestions.'
            )
            with gr.Row():
                fb_name = gr.Textbox(label='Catalyst Name', placeholder='e.g. NiCo-MXene-V')
                fb_act  = gr.Slider(0, 100, value=60, label='Measured Activity (0-100)')
                fb_stab = gr.Slider(0, 100, value=60, label='Measured Stability (0-100)')
            fb_notes  = gr.Textbox(
                label='Lab Notes',
                placeholder='e.g. coking observed at 500°C, selectivity dropped after 12 h…',
                lines=3
            )
            fb_btn      = gr.Button('Submit Experimental Result', variant='primary')
            fb_status   = gr.Markdown('_No results submitted yet._')
            feedback_chart = gr.Plot(label='Experimental Trend')

        # ── TAB 7: About ──────────────────────────────
        with gr.Tab('ℹ About'):
            gr.Markdown("""
## ⚗️ CatalystAI — Design Philosophy

| Module | What it does |
|---|---|
| **Generative Engine** | Invents novel catalyst compositions beyond known databases using Gemini 1.5 Pro reasoning |
| **ML Prediction** | Scores activity, stability, selectivity, cost on 0-100 scale |
| **Uncertainty Quantification** | Every prediction carries a calibrated confidence score (0 = certain, 1 = highly uncertain) |
| **XAI Layer** | Explains predictions using real catalytic descriptors (d-band, BET, TOF, Ea, coordination number) |
| **Pareto Optimisation** | Non-dominated sorting across all objectives; presents trade-off frontiers |
| **Self-Learning Loop** | Experimental feedback is stored and injected into every future Gemini prompt |
| **NL Interface** | Plain-language goal input auto-converts to structured system parameters |

### 🤖 AI Backend
- **Model:** Google Gemini 1.5 Pro
- **API:** Google AI Studio / Google Generative AI SDK
- **Multi-turn chat:** Native Gemini `start_chat()` session

### 🇮🇳 Aligned with India's National Priorities
- Green Hydrogen Mission
- CO₂-to-fuel conversion
- Biomass valorisation
- Atmanirbhar Bharat (earth-abundant catalyst focus)

### 🗺 Roadmap
- Materials Project & Open Catalyst Project database integration
- Molecular graph deep learning (GNN)
- Real-time lab feedback loops
- Multi-institutional collaboration workspaces

> *CatalystAI is not a database. It generates, evaluates, explains, and continuously learns.*
            """)

    # ── Wire events ──────────────────────────────────
    gen_btn.click(
        fn=run_discovery,
        inputs=[reaction_input, temp_input, press_input, solvent_input,
                excl_input, const_input, n_slider],
        outputs=[status_box, results_df, xai_md, radar_plot, pareto_plot, bar_plot, unc_plot],
    )
    btn_json.click(fn=export_json, outputs=export_status)
    btn_csv.click(fn=export_csv,  outputs=export_status)
    fb_btn.click(
        fn=submit_feedback,
        inputs=[fb_name, fb_act, fb_stab, fb_notes],
        outputs=[fb_status, feedback_chart],
    )

print('✅ UI built. Running next cell will launch it…')

/tmp/ipykernel_2891/3118604800.py:258: DeprecationWarning:

The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.



✅ UI built. Running next cell will launch it…


In [12]:
# ── CELL 6 · Launch ────────────────────────────────────────────────────────
demo.launch(
    share=True,       # creates a public gradio.live link
    debug=False,
    show_error=True,
    height=900,
)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://90192c25995a7e0ff2.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
